# 01 — Data Loading and Preprocessing

**Purpose**: Run the full GPS → pitch → active-players → transport tables pipeline
for all matches and save intermediate artefacts to `../data/` so that every
downstream figure notebook can load them without re-running the heavy pipeline.

## Pipeline
```
Raw GPS (S3)
  → pitch calibration      (src.utils.pitch_calibration)
  → match-phase labelling  (src.utils.match_phases)
  → active-player filter   (src.utils.player_status)
  → run segmentation       (src.utils.trajectory_stats)
  → collective order       (src.utils.collective_stats)
  → hazard intervals       (src.utils.hazard_stats)
  → cache as .parquet      (analysis.levy_paper.util.paper_utils)
```

## Outputs (written to `../data/`)
| File | Contents |
|------|----------|
| `runs_long.parquet` | Per-run statistics (duration, length, speed, order) |
| `trajectory_long.parquet` | Frame-level positions and velocities |
| `msd_long.parquet` | MSD decomposition (centroid, relative, player) |
| `df_pmv.parquet` | Polarisation / milling / centroid-velocity time series |
| `centroid_order_runs.parquet` | Run-level collective order summaries |
| `hazard_intervals.parquet` | Age-binned intervals for hazard modelling |

**Run this notebook once per data update; all figure notebooks load from cache.**

In [ ]:
import sys, os
from pathlib import Path

# ── repo root on path ──────────────────────────────────────────────────────
_HERE = Path(os.getcwd())
_REPO = _HERE.parents[2]          # analysis/levy_paper/notebooks → repo root
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
load_dotenv(_REPO / ".env")

from analysis.levy_paper.util.paper_utils import (
    configure_paper_plotting, save_cache,
    THETA_DEG, ACTIVE_DEPTH_M, ACTIVATE_S, BENCH_OFF_S, DATA_DIR,
)
configure_paper_plotting()
print("Repo root:", _REPO)
print("Cache dir:", DATA_DIR)

## 1. Match Selection

Load the match index and select the fixtures for this analysis.
- **Single-team matches**: Used for within-team transport and order analysis.
- **Head-to-head fixtures**: Used for coupled-team analysis (notebook 07).

> **TODO**: Add 2021 season data once uploaded to S3 (see match schedule).
> Currently using 2020 data only. Target: 4 head-to-head fixtures across 2 years.

In [ ]:
from src.utils.match_index import SoccermonMatchIndex

index = SoccermonMatchIndex()

# ── single-team sources ────────────────────────────────────────────────────
SOURCE_A = "rosenborg"     # adjust to actual source key
single_matches = index.matches_for_source(SOURCE_A)
print(f"Found {len(single_matches)} single-team matches for {SOURCE_A}")

# ── head-to-head sources ───────────────────────────────────────────────────
SOURCE_B = "valerenga"     # adjust to actual source key
h2h_fixtures = index.head_to_head(SOURCE_A, SOURCE_B)
print(f"Found {len(h2h_fixtures)} head-to-head fixtures")

display(single_matches.head())

## 2. GPS Loading and Pitch Calibration

For each match:
1. Load raw GPS (lat/lon) from S3 parquet.
2. Calibrate pitch polygon → pitch-aligned Cartesian coordinates (x_m, y_m).
3. Label match phases (pre/1H/HT/2H/post).
4. Filter to 1H and 2H only.

In [ ]:
import json
from src.utils.day_loader import DayDataLoader
from src.utils.pitch_calibration import calibrate_pitch_from_df, attach_xy_from_pitch
from src.utils.match_phases import label_match_phases
from src.utils.player_status import label_active_players

# Load pitch polygons
_pitch_json = _REPO / "notebooks" / "toppserien_pitches.json"
with open(_pitch_json) as f:
    pitch_registry = json.load(f)

loader = DayDataLoader()

all_matches_raw = {}
for _, row in single_matches.iterrows():
    match_id = row["match_id"]
    try:
        df_raw = index.load_team_match(match_id, source=SOURCE_A)
        all_matches_raw[match_id] = df_raw
    except Exception as e:
        print(f"  SKIP {match_id}: {e}")

print(f"Loaded {len(all_matches_raw)} matches")

In [ ]:
all_matches_xy = {}

for match_id, df_raw in all_matches_raw.items():
    # Calibrate pitch
    pitch_meta = calibrate_pitch_from_df(df_raw, pitch_registry)
    df_xy = attach_xy_from_pitch(df_raw, pitch_meta)

    # Label phases and keep only active play
    df_xy = label_match_phases(df_xy)
    df_play = df_xy.loc[df_xy["phase"].isin(["1H", "2H"])].copy()

    # Label active/bench players
    df_play = label_active_players(
        df_play,
        active_depth_m=ACTIVE_DEPTH_M,
        activate_s=ACTIVATE_S,
        bench_off_s=BENCH_OFF_S,
    )
    df_active = df_play.loc[df_play["player_status"] == "active"].copy()

    all_matches_xy[match_id] = df_active
    print(f"  {match_id}: {len(df_active):,} active frames, "
          f"{df_active['player_name'].nunique()} players")

## 3. Player Position Labelling

**Outstanding TODO (from original notebook)**: Add role labelling based on
distance to own goal at match start (goalkeeper, defender, midfielder, forward).

Strategy:
- At kickoff (first 5 frames of 1H), compute each player's median x_m.
- Assign positional quartile: GK (deepest), DEF, MID, FWD.
- Store as `position_label` column — useful for conditioning analyses in
  notebooks 03 and 06.

In [ ]:
def assign_position_labels(df_active: pd.DataFrame, n_kickoff_frames: int = 5) -> pd.DataFrame:
    """
    Assign a coarse positional label (GK/DEF/MID/FWD) to each player
    based on their depth (x_m) at match kickoff.

    Parameters
    ----------
    df_active       : Active-players dataframe with x_m, phase, player_name.
    n_kickoff_frames: Number of 1H frames to use as the 'kickoff window'.

    Returns
    -------
    df_active with a new 'position_label' column.
    """
    # Kickoff window = first n frames of 1H
    ko = (
        df_active.loc[df_active["phase"] == "1H"]
        .sort_values("timestamp")
        .groupby("player_name")
        .head(n_kickoff_frames)
    )

    depth = ko.groupby("player_name")["x_m"].median().rename("depth_m")

    # Quartile-based labels (team-relative: most negative x_m = closest to own goal)
    q25, q50, q75 = depth.quantile([0.25, 0.50, 0.75])

    def _label(d: float) -> str:
        if d < q25:
            return "GK/DEF"
        elif d < q50:
            return "DEF/MID"
        elif d < q75:
            return "MID/FWD"
        return "FWD"

    pos_map = depth.apply(_label).to_dict()
    df_active = df_active.copy()
    df_active["position_label"] = df_active["player_name"].map(pos_map).fillna("Unknown")
    return df_active


# Apply to all matches
for mid in list(all_matches_xy.keys()):
    all_matches_xy[mid] = assign_position_labels(all_matches_xy[mid])

# Sanity check
sample = next(iter(all_matches_xy.values()))
print(sample.groupby("position_label")["player_name"].nunique())

## 4. Transport Tables

Build run-level and frame-level transport statistics for all matches.
This is the most compute-intensive step (~1–2 min per match).

In [ ]:
from src.utils.trajectory_stats import build_transport_tables_from_active_v2

all_transport = {}

for match_id, df_active in all_matches_xy.items():
    print(f"Building transport tables: {match_id} …", end=" ")
    transport = build_transport_tables_from_active_v2(
        df_active,
        theta_deg=THETA_DEG,
    )
    all_transport[match_id] = transport
    print("done")

# Concatenate across matches
runs_long       = pd.concat([t["runs_long"]       for t in all_transport.values()], ignore_index=True)
trajectory_long = pd.concat([t["trajectory_long"] for t in all_transport.values()], ignore_index=True)
msd_long        = pd.concat([t["msd_long"]        for t in all_transport.values()], ignore_index=True)

print(f"\nRuns: {len(runs_long):,} | Frames: {len(trajectory_long):,}")

## 5. Collective Order Tables

Compute polarisation p(t), milling m(t), and centroid velocity v_c(t)
for each match, then attach run-level order summaries.

Three order measures per run:
- `p_mean`  — retrospective mean (for descriptive stats only, not as predictor)
- `p_start` — polarisation at run start (predictive-safe)
- `p_early_3s` — mean over first 3 seconds (predictive-safe)

In [ ]:
from src.utils.collective_stats import build_collective_order_tables_from_transport
from analysis.levy_paper.util.paper_utils import assign_order_state, STATE_BOUNDS, STATE_LABELS

all_pmv          = {}
all_centroid_ord = {}

for match_id, transport in all_transport.items():
    df_pmv, centroid_order_runs = build_collective_order_tables_from_transport(transport)
    all_pmv[match_id]          = df_pmv
    all_centroid_ord[match_id] = centroid_order_runs

df_pmv           = pd.concat(all_pmv.values(), ignore_index=True)
centroid_ord_runs = pd.concat(all_centroid_ord.values(), ignore_index=True)

# Assign state labels
centroid_ord_runs["order_state"] = assign_order_state(
    centroid_ord_runs["p_mean"],
    bounds=STATE_BOUNDS,
    labels=STATE_LABELS,
)

print(centroid_ord_runs["order_state"].value_counts())

## 6. Hazard Intervals

Discretise each run into age-bins (1 s intervals) with:
- `event` = 1 if the run terminates within this interval, 0 otherwise
- `p_state` = order state at the interval midpoint
- `speed_mps` = mean centroid speed in the interval

In [ ]:
from src.utils.hazard_stats import build_centroid_hazard_intervals

all_haz = {}
for match_id, transport in all_transport.items():
    df_pmv_m = all_pmv[match_id]
    haz = build_centroid_hazard_intervals(transport, df_pmv_m)
    all_haz[match_id] = haz

hazard_intervals = pd.concat(all_haz.values(), ignore_index=True)
print(f"Hazard intervals: {len(hazard_intervals):,}  |  events: {hazard_intervals['event'].sum():,}")

## 7. Save All Caches

Write all artefacts to `../data/` as parquet files.

In [ ]:
from analysis.levy_paper.util.paper_utils import save_cache

artefacts = {
    "runs_long":           runs_long,
    "trajectory_long":     trajectory_long,
    "msd_long":            msd_long,
    "df_pmv":              df_pmv,
    "centroid_order_runs": centroid_ord_runs,
    "hazard_intervals":    hazard_intervals,
}

for name, df in artefacts.items():
    path = save_cache(df, name)
    print(f"  ✓  {name:30s}  →  {path.name}  ({len(df):,} rows)")

print("\nAll caches saved. Run downstream notebooks.")

## 8. Quick Diagnostic Plots

Fast sanity checks — not paper figures.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Run duration histogram
axes[0].hist(runs_long["duration_s"].clip(upper=120), bins=50, edgecolor="k", lw=0.3)
axes[0].set_xlabel("Duration (s)"); axes[0].set_title("Run durations")

# Polarisation histogram
axes[1].hist(df_pmv["p_group"].dropna(), bins=50, edgecolor="k", lw=0.3, color="#4393c3")
axes[1].set_xlabel("Polarisation p"); axes[1].set_title("Polarisation distribution")

# Hazard event rate vs age
haz_diag = (
    hazard_intervals
    .groupby("age_bin")["event"]
    .agg(["sum", "count"])
    .assign(rate=lambda d: d["sum"] / d["count"])
)
axes[2].plot(haz_diag.index, haz_diag["rate"], "k-o", ms=3)
axes[2].set_xlabel("Age bin (s)"); axes[2].set_title("Empirical hazard rate")

plt.tight_layout()
plt.show()